# Deep Learning 010 — Forward Propagation

The whole forward pass is one loop with two lines in it. This notebook builds the
4-3-2-1 network from the lesson, checks every shape, counts every parameter, and shows
the batched version is the same arithmetic.

In [ ]:
import numpy as np

def sigmoid(z):
    return 1 / (1 + np.exp(-np.clip(z, -60, 60)))

def init(layer_dims, seed=0):
    rng = np.random.default_rng(seed)
    params = {}
    for l in range(1, len(layer_dims)):
        params[f'W{l}'] = rng.normal(size=(layer_dims[l-1], layer_dims[l])) * 0.5
        params[f'b{l}'] = np.zeros(layer_dims[l])
    return params

DIMS = [4, 3, 2, 1]
params = init(DIMS)
for k, v in params.items():
    print(f'{k:>4} shape {v.shape}')

## Five lines

`A` starts as the input and is overwritten layer by layer. That reassignment *is* the
forward pass — the output of one layer is the input of the next, and nothing else
happens.

In [ ]:
def forward(X, params, n_layers, keep=False):
    A = X
    cache = [A]
    for l in range(1, n_layers + 1):
        Z = A @ params[f'W{l}'] + params[f'b{l}']    # linear
        A = sigmoid(Z)                                # nonlinear
        if keep:
            cache.append(A)
    return (A, cache) if keep else A

x = np.array([[0.5, -1.2, 0.3, 0.8]])          # one row, 4 features
out, cache = forward(x, params, len(DIMS) - 1, keep=True)
for i, a in enumerate(cache):
    print(f'layer {i} activations shape {a.shape}  {np.round(a, 4)}')
print(f'\noutput {out.item():.6f}')

## Verify every shape against the arithmetic

If a shape is wrong the code still runs — NumPy broadcasting is generous — and the
answer is silently nonsense. So assert instead of eyeballing.

In [ ]:
for l in range(1, len(DIMS)):
    assert params[f'W{l}'].shape == (DIMS[l-1], DIMS[l]), l
    assert params[f'b{l}'].shape == (DIMS[l],), l
assert out.shape == (1, 1)
print('every shape checks out')

total = sum(v.size for v in params.values())
by_hand = 4*3 + 3 + 3*2 + 2 + 2*1 + 1
print(f'\nparameters: counted {total}, by hand {by_hand}, match={total == by_hand}')
print('  4*3 + 3 = 15   3*2 + 2 = 8   2*1 + 1 = 3')

## One row or a thousand — the same two lines

Nothing in `forward` mentions the batch size. Feed it 1,000 rows and the shapes just
carry through.

In [ ]:
rng = np.random.default_rng(1)
batch = rng.normal(size=(1000, 4))
out_batch = forward(batch, params, 3)
print('batch output shape', out_batch.shape)

# And row i of the batch equals the single-row result for that row.
single = np.vstack([forward(batch[i:i+1], params, 3) for i in range(20)])
assert np.allclose(single, out_batch[:20])
print('first 20 rows match the one-at-a-time computation')
print('\nThis is why GPUs matter: one matrix multiply for 1,000 rows, not 1,000 of them.')

> **TensorFlow is optional here.** Every cell above runs on NumPy alone. The Keras
> comparison below is the check that your hand count is right — if TensorFlow is not
> installed the cell skips itself with a message instead of failing.

In [ ]:
try:
    from tensorflow import keras
    import numpy as np
    m = keras.Sequential([
        keras.layers.Input(shape=(4,)),
        keras.layers.Dense(3, activation='sigmoid'),
        keras.layers.Dense(2, activation='sigmoid'),
        keras.layers.Dense(1, activation='sigmoid')])
    m.set_weights([params['W1'], params['b1'],
                   params['W2'], params['b2'],
                   params['W3'], params['b3']])
    keras_out = m.predict(x, verbose=0)
    print(f'ours  {out.item():.8f}')
    print(f'keras {keras_out.item():.8f}')
    print(f'match {np.allclose(out, keras_out)}')
    m.summary()
except ImportError:
    print('TensorFlow not installed - skipping the Keras cross-check.')
    print('Note the weight ORDER Keras expects: [W1, b1, W2, b2, ...], with W')
    print('shaped (n_in, n_out) exactly as above. Run this cell later to confirm')
    print('the two implementations agree to eight decimal places.')

## Exercises

1. Replace `sigmoid` with `relu` in the hidden layers only, keeping sigmoid on the
   output. What changes in the activations?
2. Remove the nonlinearity entirely (`A = Z`). Show that the resulting 4-3-2-1 network
   is equivalent to a single 4→1 linear layer by finding the one matrix that reproduces
   it. This is *why* activations are not optional.
3. Time `forward` on 1 row versus 100,000 rows. Is it 100,000 times slower? Explain.
4. Add a `n_layers` of your choosing and rebuild for `[10, 8, 6, 4, 2, 1]`. Count the
   parameters by hand first.